# Nika on Colab

Train the ~30M model on a bigger slice of FineWeb-Edu.

**Before running:** Runtime -> Change runtime type -> GPU (T4).

What lives where:

| | where | why |
|---|---|---|
| code | cloned from GitHub | 5 seconds, and git stays the source of truth |
| tiny.txt (2 GB) | `/content` | regenerating it is only a download |
| train.bin / val.bin | **Drive**, copied to `/content` | 25 min to rebuild, but too slow to train from over the network |
| checkpoints | `/content`, copied to **Drive** every 5 evals | written often, and Drive is slow for a 360 MB file |

After a disconnect, rerun every cell. Each one skips work that is already done, and training resumes from the last checkpoint.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch; print(torch.__version__, torch.cuda.is_available(), 'bf16:', torch.cuda.is_bf16_supported())

In [ ]:
# clone (or update) the repo and work from its root, so `python -m scripts.x` imports work
import os, shutil
REPO = "https://github.com/dat999zx/nika.git"
if not os.path.exists("/content/nika"):
    !git clone $REPO /content/nika
%cd /content/nika
!git pull
!pip -q install datasets

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/nika'   # permanent: data cache + checkpoint backups
LOCAL_CKPT = '/content/ckpt/nika30m.pt' # fast local disk, what training writes to
os.makedirs(DRIVE, exist_ok=True)
os.makedirs('/content/ckpt', exist_ok=True)
print(DRIVE, LOCAL_CKPT)

In [ ]:
# Data: reuse the tokenized .bin files from Drive if they are there, otherwise
# download + encode once and cache them. Encoding uses the committed 4096 tokenizer.
os.makedirs('data', exist_ok=True)
have_cache = all(os.path.exists(f'{DRIVE}/{n}') for n in ('train.bin', 'val.bin'))

if have_cache:
    for n in ('train.bin', 'val.bin'):
        if not os.path.exists(f'data/{n}'):
            print('copying', n, 'from Drive...')
            shutil.copy2(f'{DRIVE}/{n}', f'data/{n}')
else:
    !python -m scripts.download_data --mb 2000
    !python -m scripts.encode_data
    for n in ('train.bin', 'val.bin'):
        shutil.copy2(f'data/{n}', f'{DRIVE}/{n}')   # cache for the next session

!ls -la data/*.bin

In [ ]:
# ~30M params: n_embed 512, 6 layers, 8 heads (head_size 64), context 256.
# Writes checkpoints to /content every eval, copies them to Drive every 5 evals.
# --resume restores from Drive first if the local disk is empty, so rerunning this
# cell after a disconnect continues where it stopped.
#
# Started in the BACKGROUND so the next cell can show the report while it trains.
# It also means interrupting this cell does not kill training.
import subprocess

LOG = '/content/train.log'
cmd = f"""python -m scripts.train \
    --n-embed 512 --n-layer 6 --n-head 8 --block-size 256 \
    --batch-size 48 --max-iters 20000 --lr 6e-4 \
    --eval-interval 500 --eval-iters 40 --warmup-iters 500 \
    --checkpoint {LOCAL_CKPT} --backup-dir {DRIVE} --backup-every 5 --resume"""

log = open(LOG, 'w')
proc = subprocess.Popen(cmd, shell=True, stdout=log, stderr=subprocess.STDOUT, cwd='/content/nika')
print('training started, pid', proc.pid)

In [ ]:
# Live report: refreshes every 30s while training runs, so you can watch the loss
# table and curve fill in. Interrupting this cell only stops the display, not training.
import time
from IPython.display import Markdown, Image, clear_output, display

REPORT = LOCAL_CKPT.replace('.pt', '_report.md')
PLOT = LOCAL_CKPT.replace('.pt', '_loss.png')

while True:
    running = proc.poll() is None
    clear_output(wait=True)
    if os.path.exists(REPORT):
        # the markdown image link does not resolve in Colab, so drop that section
        # and show the png itself instead
        display(Markdown(open(REPORT).read().split('## Loss curve')[0]))
        if os.path.exists(PLOT):
            display(Image(PLOT))
    print(''.join(open(LOG).readlines()[-8:]))  # tail of the training log
    if not running:
        print('training finished, exit code', proc.returncode)
        break
    time.sleep(30)

In [ ]:
!python -m scripts.generate --checkpoint $LOCAL_CKPT --prompt "Photosynthesis is the process by which plants" --tokens 150 --temperature 0.8